## Day 12 — Cost Optimization Basics

**What we learn:** How to measure and reduce unnecessary Spark work.

**What works on Free Edition serverless:**
- ✅ Job runtime analysis with `time.time()`
- ✅ Reducing unnecessary `.count()` / `.show()` actions
- ✅ Column pruning with `.select()`
- ✅ Predicate pushdown with early `.filter()`
- ✅ `DESCRIBE DETAIL` — inspect table partition structure

**Theory only (not available on serverless):**
- 📚 Cluster sizing — Databricks manages this automatically
- 📚 `df.cache()` / `cacheTable()` — blocked on serverless
- 📚 `partitionBy()` on write — not supported in `writeTo()`

**Key rule:** Every `.count()`, `.show()`, `.collect()` on a full DataFrame = one full table scan.

In [0]:
import time
from pyspark.sql.functions import col, count, avg, round as spark_round, desc

EVENTS = 'ecommerce.silver.events_si'
PREDS  = 'ecommerce.gold.user_predictions'
print('Ready')


Ready


### Task 1 — Analyze Job Runtime

First measure the baseline: how long does a clean aggregation take with no wasted work?

In [0]:
# Baseline — clean query, no wasted actions
start = time.time()

result = (
    spark.table(EVENTS)
    .groupBy('event_type')
    .agg(
        count('*').alias('events'),
        spark_round(avg('price'), 2).alias('avg_price')
    )
    .orderBy(desc('events'))
)
result.show()

t_baseline = time.time() - start
print(f'Baseline time: {t_baseline}s')


+----------+---------+---------+
|event_type|   events|avg_price|
+----------+---------+---------+
|      view|104318354|   291.11|
|      cart|  3828450|    300.8|
|  purchase|  1659703|   304.35|
+----------+---------+---------+

Baseline time: 38.253623723983765s


### Costly pattern — unnecessary `.count()` calls

A common mistake: calling `.count()` to log progress at each step of a pipeline. Each `.count()` on a 5.7M row table is a **full table scan**. Three `.count()` calls = three full scans.

In [0]:
# BAD pattern — three .count() calls on the same table
# Each one triggers a full scan from scratch
start = time.time()

df = spark.table(EVENTS)

# Scan 1 — logging total rows (unnecessary)
n_total = df.count()
print(f'Total rows: {n_total:,}')

# Scan 2 — logging purchase rows (unnecessary)
n_purchase = df.filter(col('event_type') == 'purchase').count()
print(f'Purchase rows: {n_purchase:,}')

# Scan 3 — the actual result we needed
result_costly = (
    df.groupBy('event_type')
    .agg(count('*').alias('events'), spark_round(avg('price'), 2).alias('avg_price'))
    .orderBy(desc('events'))
)
result_costly.show()

t_costly = time.time() - start
print(f'Costly time: {t_costly}s  ({3} full scans)')


Total rows: 109,806,507
Purchase rows: 1,659,703
+----------+---------+---------+
|event_type|   events|avg_price|
+----------+---------+---------+
|      view|104318354|   291.11|
|      cart|  3828450|    300.8|
|  purchase|  1659703|   304.35|
+----------+---------+---------+

Costly time: 4.824687480926514s  (3 full scans)


### Task 2 — Reduce Unnecessary Actions

Rewrite the same logic using **one scan** — get all the information from a single `groupBy` instead of three separate queries.

In [0]:
# GOOD pattern — one groupBy gives us everything we need
# Total rows, purchase count, AND the aggregation — all from one scan
start = time.time()

result_lean = (
    spark.table(EVENTS)
    .groupBy('event_type')
    .agg(
        count('*').alias('events'),
        spark_round(avg('price'), 2).alias('avg_price')
    )
    .orderBy(desc('events'))
)
result_lean.show()

# Total and purchase count come FREE from the groupBy result
rows = result_lean.collect()
total   = sum(r['events'] for r in rows)
purchase = next((r['events'] for r in rows if r['event_type']=='purchase'), 0)
print(f'Total rows: {total:,}')
print(f'Purchase rows: {purchase:,}')

t_lean = time.time() - start
print(f'Lean time: {t_lean}s  (1 scan)')


+----------+---------+---------+
|event_type|   events|avg_price|
+----------+---------+---------+
|      view|104318354|   291.11|
|      cart|  3828450|    300.8|
|  purchase|  1659703|   304.35|
+----------+---------+---------+

Total rows: 109,806,507
Purchase rows: 1,659,703
Lean time: 2.5898654460906982s  (1 scan)


In [0]:
print('=== JOB RUNTIME COMPARISON ===')
print(f'Baseline (1 scan, show):      {t_baseline}s')
print(f'Costly   (3 scans, 3 counts): {t_costly}s')
print(f'Lean     (1 scan, groupBy):   {t_lean}s')
print()
saved = t_costly - t_lean
print(f'Time saved by removing unnecessary scans: {saved}s')
print()
print('On a paid cluster: time saved = money saved.')
print('On free tier: time saved = daily quota preserved.')


=== JOB RUNTIME COMPARISON ===
Baseline (1 scan, show):      38.253623723983765s
Costly   (3 scans, 3 counts): 4.824687480926514s
Lean     (1 scan, groupBy):   2.5898654460906982s

Time saved by removing unnecessary scans: 2.2348220348358154s

On a paid cluster: time saved = money saved.
On free tier: time saved = daily quota preserved.


### Column Pruning — only read what you need

If your table has 20 columns but your query only needs 3, use `.select()` to tell Spark to skip the other 17. Parquet stores columns separately — reading 3 columns is much faster than reading 20.

In [0]:
# Compare: full table scan vs column-pruned scan

# Without pruning — reads ALL columns from parquet
start = time.time()
spark.table(PREDS).groupBy('predicted_buyer').count().show()
t_full = time.time() - start

# With pruning — reads ONLY the one column we need
start = time.time()
spark.table(PREDS).select('predicted_buyer').groupBy('predicted_buyer').count().show()
t_pruned = time.time() - start

print(f'Without .select(): {t_full}s   (reads all columns)')
print(f'With .select():    {t_pruned}s   (reads 1 column only)')
print()
print('Photon + Delta are already quite good at column pruning automatically.')
print('But explicit .select() makes our intent clear and helps the optimizer.')


+---------------+-------+
|predicted_buyer|  count|
+---------------+-------+
|           true|1316822|
|          false|4404776|
+---------------+-------+

+---------------+-------+
|predicted_buyer|  count|
+---------------+-------+
|           true|1316822|
|          false|4404776|
+---------------+-------+

Without .select(): 1.634643316268921s   (reads all columns)
With .select():    0.8035414218902588s   (reads 1 column only)

Photon + Delta are already quite good at column pruning automatically.
But explicit .select() makes our intent clear and helps the optimizer.


### Predicate Pushdown — filter early, read less

Apply `.filter()` as early as possible in your pipeline — before joins, before groupBy. Spark pushes the filter into the file reader so it never loads the rows you are going to throw away anyway.

In [0]:
# Filter LATE — loads all rows then filters
start = time.time()
(
    spark.table(EVENTS)
    .groupBy('event_type', 'product_id')
    .agg(count('*').alias('n'))
    .filter(col('event_type') == 'purchase')   # filter AFTER groupBy
    .orderBy(desc('n'))
).show(5)
t_late = time.time() - start

# Filter EARLY — pushes filter into file scan, loads only purchase rows
start = time.time()
(
    spark.table(EVENTS)
    .filter(col('event_type') == 'purchase')   # filter BEFORE groupBy
    .groupBy('product_id')
    .agg(count('*').alias('n'))
    .orderBy(desc('n'))
).show(5)
t_early = time.time() - start

print(f'Filter late:  {t_late}s')
print(f'Filter early: {t_early}s')
print()
print('Check .explain() — early filter shows PushedFilters in PhotonScan')


+----------+----------+-----+
|event_type|product_id|    n|
+----------+----------+-----+
|  purchase|   1004856|61259|
|  purchase|   1004767|44417|
|  purchase|   1005115|34785|
|  purchase|   4804056|30179|
|  purchase|   1004833|26183|
+----------+----------+-----+
only showing top 5 rows
+----------+-----+
|product_id|    n|
+----------+-----+
|   1004856|61259|
|   1004767|44417|
|   1005115|34785|
|   4804056|30179|
|   1004833|26183|
+----------+-----+
only showing top 5 rows
Filter late:  1.7318017482757568s
Filter early: 0.9818768501281738s

Check .explain() — early filter shows PushedFilters in PhotonScan


### Inspect partition structure with DESCRIBE DETAIL

`DESCRIBE DETAIL` shows whether a table is partitioned, how many files it has, and how large they are. This tells you if partition pruning is possible.

In [0]:
# DESCRIBE DETAIL shows partition info, file count, table size
print('=== events_si table structure ===')
spark.sql(f'DESCRIBE DETAIL {EVENTS}').select(
    'format', 'numFiles', 'sizeInBytes', 'partitionColumns'
).show(truncate=False)

print('=== user_predictions table structure ===')
spark.sql(f'DESCRIBE DETAIL {PREDS}').select(
    'format', 'numFiles', 'sizeInBytes', 'partitionColumns'
).show(truncate=False)

# What to look for:
# partitionColumns = [] -> table is NOT partitioned -> no pruning possible
# partitionColumns = ['event_type'] -> partitioned -> pruning works
# numFiles -> many small files = slow (small file problem)
# sizeInBytes -> divide by numFiles to get avg file size


=== events_si table structure ===
+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |61      |3701076747 |[event_date]    |
+------+--------+-----------+----------------+

=== user_predictions table structure ===
+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |2       |21213007   |[]              |
+------+--------+-----------+----------------+



### Task 3 — Cost-Saving Audit of your pipeline

In [0]:
print('=== COST-SAVING AUDIT: 9-Day Pipeline ===')
print()

audit = [
    ('Day 3',  'Silver table creation',
     'Low',    'writeTo().createOrReplace() — one scan, no wasted actions'),
    ('Day 5',  'ML dataset with .count() logging',
     'Medium', 'Every .count() is a full scan on 5.7M rows — remove progress counts'),
    ('Day 6',  'Model training with evaluator',
     'Low',    'transform() + evaluate() — two scans needed, cannot reduce'),
    ('Day 7',  'MLflow logging with sample',
     'Low',    '100-row sample for signature — minimal cost'),
    ('Day 8',  'Batch inference on 5.7M users',
     'High',   'model.transform() is one pass — good. But display() after write = extra scan'),
    ('Day 9',  'ALS cross join for recommendations',
     'High',   'Cross join users x products = most expensive operation in pipeline'),
    ('Day 10', 'Query optimization timing',
     'Medium', '3-run timing loop = 3 full scans — acceptable for learning, remove in production'),
    ('Day 11', 'Time travel comparisons',
     'Medium', 'df_before.count() + df_current.count() = 2 scans — combine into one groupBy'),
]

print(f'{"Day":<8} {"Task":<35} {"Cost":<8} {"Fix"}')
print('-' * 100)
for day, task, cost, fix in audit:
    print(f'{day:<8} {task:<35} {cost:<8} {fix}')

print()
print('Top 3 cost-saving actions for this pipeline:')
print('  1. Remove .count() calls used only for logging — saves 1 scan per call')
print('  2. Replace display(df) with display(df.limit(10)) — no full scan')
print('  3. Day 9 ALS cross join: limit users to high-probability buyers from Day 8')
print('     instead of scoring all 5.7M — reduces cross join size by ~85%')


=== COST-SAVING AUDIT: 9-Day Pipeline ===

Day      Task                                Cost     Fix
----------------------------------------------------------------------------------------------------
Day 3    Silver table creation               Low      writeTo().createOrReplace() — one scan, no wasted actions
Day 5    ML dataset with .count() logging    Medium   Every .count() is a full scan on 5.7M rows — remove progress counts
Day 6    Model training with evaluator       Low      transform() + evaluate() — two scans needed, cannot reduce
Day 7    MLflow logging with sample          Low      100-row sample for signature — minimal cost
Day 8    Batch inference on 5.7M users       High     model.transform() is one pass — good. But display() after write = extra scan
Day 9    ALS cross join for recommendations  High     Cross join users x products = most expensive operation in pipeline
Day 10   Query optimization timing           Medium   3-run timing loop = 3 full scans — acceptable f

## Day 12 Complete!

| Cell | What it does |
|------|--------------|
| 2 | **Task 1** — Baseline runtime: 1 clean scan |
| 3 | Costly pattern: 3 unnecessary `.count()` calls |
| 4 | Lean pattern: same result from 1 `groupBy` scan |
| 5 | Side-by-side time comparison |
| 6 | Column pruning with `.select()` |
| 7 | **Task 2** — Predicate pushdown: filter early vs filter late |
| 8 | `DESCRIBE DETAIL` — check partition structure |
| 9 | **Task 3** — Cost-saving audit of the full 9-day pipeline |

**Three rules for serverless cost optimization:**
1. Every `.count()` = one full scan — remove unless essential
2. `.select()` only the columns you need
3. `.filter()` as early as possible — before joins and groupBy